# Step 4 — Gender annotation schema v0.2: machine-readable schema and crosswalk from the earlier silver labels

**Purpose.** This notebook records the schema adopted in `annotation-guidelines-gender-v0.2.md` (adapted from Bórquez and Mendoza's annotation guide, with the approved decisions D1–D5) in machine-readable form, and shows how the earlier Step 8e silver labels map onto it. It exists so the new schema is kept separate from the older Step 8 notebooks, which use the older Layer A/B labels.

**Inputs**
- `outputs/analysis/units.csv` — the offset-preserving units from the Steps 1–3 notebook (7,971 units).
- `outputs/RESTRICTED_raw_and_contact/raw_layer.csv` — read only to verify raw-text offsets; no raw text is written out.
- `../outputs/equal_step8e_layerA_rule_detection/step8e_peso_layerA_span_annotations.csv` — the earlier rule-based (silver) span annotations.
- `annotation-guidelines-gender-v0.2.md` — checked for consistency with the schema defined here.

**Outputs** (`outputs/s4_schema_v0.2/`)
- `schema_v0.2.json` — labels, families, whether each counts as bias, origin.
- `crosswalk_step8e_to_v0.2.csv` — one row per Step 8e span, with a **provisional** v0.2 label, a review flag and the span's raw character offsets.
- `crosswalk_summary.csv`, `validation_checks.csv`.

**What this is not.** The v0.2 labels in the crosswalk are deterministic translations of rule output. They are candidates for human review, not annotations. No human has labelled any unit, no agreement has been measured, and no model is trained here.

## 1. Load inputs and check that the earlier labels align with the new units

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)

BASE = Path(".")
UNITS_PATH = BASE / "outputs" / "analysis" / "units.csv"
RAW_PATH = BASE / "outputs" / "RESTRICTED_raw_and_contact" / "raw_layer.csv"
SPANS_8E = Path("..") / "outputs" / "equal_step8e_layerA_rule_detection" / "step8e_peso_layerA_span_annotations.csv"
GUIDE_PATH = BASE / "annotation-guidelines-gender-v0.2.md"
OUT = BASE / "outputs" / "s4_schema_v0.2"
OUT.mkdir(parents=True, exist_ok=True)

units = pd.read_csv(UNITS_PATH, encoding="utf-8-sig", dtype={"mask_edits": str}).set_index("unit_id", drop=False)
raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False).set_index("record_id")
s8e = pd.read_csv(SPANS_8E, encoding="utf-8-sig")
print(f"{len(units):,} units; {len(s8e):,} Step 8e span rows covering {s8e['unit_id'].nunique():,} units.")

compact = lambda t: re.sub(r"\s+", " ", str(t)).strip()
joined = s8e.merge(units[["text_masked", "component"]].rename(columns={"component": "component_new"}), left_on="unit_id", right_index=True, how="left")
print("Step 8e unit_ids missing from the new units :", int(joined["text_masked"].isna().sum()))
print("Step 8e sentence differs from new unit text :", int((joined["sentence"].map(compact) != joined["text_masked"].map(compact)).sum()))
print("Step 8e component differs from new component:", int((joined["component"] != joined["component_new"]).sum()))

7,971 units; 8,115 Step 8e span rows covering 7,971 units.
Step 8e unit_ids missing from the new units : 0
Step 8e sentence differs from new unit text : 0
Step 8e component differs from new component: 0


## 2. The v0.2 schema

Defined once here and written to JSON. `counts_as_bias` follows the guideline's rule: E1–E5 explicit, I1–I2 implicit; INC (inclusive) and P (polisemantic) are not bias; O (others) is not counted until reviewed.

In [2]:
SCHEMA = [
    # code, name, family, counts_as_bias, origin
    ("E1", "Gender Restriction / Preference", "explicit", True, "extension (D1)"),
    ("E2", "Explicit Marking of Sex", "explicit", True, "reference"),
    ("E3", "Gendered Neologism", "explicit", True, "reference"),
    ("E4", "Generic He", "explicit", True, "reference"),
    ("E5", "Generic She", "explicit", True, "reference"),
    ("I1", "Masculine Coded", "implicit", True, "reference"),
    ("I2", "Feminine Coded", "implicit", True, "reference"),
    ("INC", "Inclusive", "not_bias", False, "extension (D3, changed by user)"),
    ("P", "Polisemantic", "not_bias", False, "reference"),
    ("O", "Others", "pending_review", False, "reference"),
]
schema_df = pd.DataFrame(SCHEMA, columns=["code", "name", "family", "counts_as_bias", "origin"])
display(schema_df)

DECISIONS = {
    "D1": "E1 Gender Restriction/Preference added as an own label",
    "D2": "Gendered titles (foreman, salesman, watchman) are E2, explicit",
    "D3": "Female/Male style listings are INC (inclusive), not bias, no rewrite",
    "D4": "Honorifics (Ms./Mr.) for contact persons are P, not bias",
    "D5": "Gender-differentiated requirements and pregnancy are O with a note",
}
(OUT / "schema_v0.2.json").write_text(
    json.dumps(
        {
            "version": "0.2",
            "source": "adapted from Borquez and Mendoza, Annotation Guidelines: Bias in Job Advertisement",
            "status": "draft; not piloted; no human labels",
            "labels": schema_df.to_dict(orient="records"),
            "decisions": DECISIONS,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

guide = GUIDE_PATH.read_text(encoding="utf-8")
missing_in_guide = [c for c in schema_df["code"] if f"**{c}**" not in guide]
print("Codes missing from the guideline document:", missing_in_guide or "none")
print("Obsolete code B1 / 'Binary Listing' still in the guideline:", ("B1" in guide) or ("Binary Listing" in guide))

,code,name,family,counts_as_bias,origin
0,E1,Gender Restriction / Preference,explicit,True,extension (D1)
1,E2,Explicit Marking of Sex,explicit,True,reference
2,E3,Gendered Neologism,explicit,True,reference
3,E4,Generic He,explicit,True,reference
4,E5,Generic She,explicit,True,reference
5,I1,Masculine Coded,implicit,True,reference
6,I2,Feminine Coded,implicit,True,reference
7,INC,Inclusive,not_bias,False,"extension (D3, changed by user)"
8,P,Polisemantic,not_bias,False,reference
9,O,Others,pending_review,False,reference


Codes missing from the guideline document: none
Obsolete code B1 / 'Binary Listing' still in the guideline: False


## 3. Crosswalk rules (Step 8e silver → v0.2 candidate)

Each Step 8e span row is translated by a fixed rule. A row the rule cannot settle is given a candidate label together with a review flag. These rules mirror §3 and §7 of the guideline; they are not trained.

| Step 8e expression type | v0.2 candidate | Condition / flag |
|---|---|---|
| restriction, preference | **E1** | span contains a gender word; otherwise `reject_candidate_no_gender_word` |
| restriction in a job title with a bare gendered word (e.g. "Boy") | **E2** | title wording, not a requirement |
| binary_listing | **INC** | includes "He/She" |
| gendered_title_pronoun, pronoun span (he, she, his, her, him) | `E4_or_E5` | needs human context (generic vs a named person) |
| gendered_title_pronoun, other span | **E2** | gendered title |
| gender_coded, agentic / communal | **I1 / I2** | flag `P_candidate_role_noun` when the span is analyst, analysis, support or leader (role nouns named in guideline §3) |
| unclear, span "Ms" | **P** | honorific (D4) |
| unclear, other | `needs_review` | |
| E1 span longer than 3 words | | flag `span_narrowing_review` (guideline §4: span = gender-bearing words) |

In [3]:
GENDER_WORD = re.compile(r"\b(?:male|female|males|females|men|women|man|woman|boy|girl|lady|ladies|gentleman|lalaki|babae)\b", re.I)
PRONOUNS = {"he", "she", "his", "her", "him"}
BARE_TITLE_WORDS = {"boy", "girl", "lady", "man", "woman"}
ROLE_NOUNS = {"analyst", "analysis", "support", "leader"}


def to_v02(row):
    t = row["gender_expression_type"]
    span = "" if pd.isna(row["span_text"]) else str(row["span_text"])
    low = span.strip().lower()
    flag, reason = "", ""
    if t in ("restriction", "preference"):
        if not GENDER_WORD.search(span):
            return "none", "reject_candidate_no_gender_word", "restriction/preference rule matched a span without a gender word"
        if row["component"] == "job_title" and low in BARE_TITLE_WORDS:
            return "E2", "", "gendered word inside a job title"
        if len(span.split()) > 3:
            flag = "span_narrowing_review"
        return "E1", flag, "stated gender restriction or preference"
    if t == "binary_listing":
        return "INC", "", "both genders listed"
    if t == "gendered_title_pronoun":
        if low in PRONOUNS:
            return "E4_or_E5", "needs_human_context", "pronoun: generic or about a named person"
        return "E2", "", "gender-marked job title"
    if t == "gender_coded":
        code = {"agentic": "I1", "communal": "I2"}.get(str(row["gender_coding"]).lower(), "needs_review")
        if low in ROLE_NOUNS:
            flag = "P_candidate_role_noun"
        return code, flag, "lexicon-coded wording"
    if t == "unclear":
        if low == "ms":
            return "P", "honorific_candidate", "honorific for a contact person"
        return "needs_review", "needs_human_context", "gender word the rules could not classify"
    return "none", "", ""


spans = s8e[s8e["gender_expression_type"] != "none"].copy()
res = spans.apply(to_v02, axis=1, result_type="expand")
spans["v02_candidate"], spans["review_flag"], spans["crosswalk_reason"] = res[0], res[1], res[2]
print(f"{len(spans):,} Step 8e span rows with a gender expression ({spans['unit_id'].nunique():,} units).")
display(spans["v02_candidate"].value_counts().rename_axis("v0.2 candidate").to_frame("span rows"))

1,218 Step 8e span rows with a gender expression (1,074 units).


,span rows
v0.2 candidate,
I2,379
I1,350
E1,317
INC,99
E2,40
needs_review,16
P,11
E4_or_E5,5
none,1


## 4. Raw character offsets for every span

Step 8e span offsets index the unit text. They are converted to raw-advertisement offsets with the unit's `char_start` and `mask_edits` (the offset map produced in Steps 1–3). A span whose Step 8e offsets do not reproduce its own text is flagged and given no raw offsets.

In [4]:
def masked_span_to_raw(edits, char_start, a, b):
    pos_m, pos_r = 0, char_start
    result = {}
    for rs, re_, ph in edits:
        gap = rs - pos_r
        for key, x in (("a", a), ("b", b)):
            if key not in result and x <= pos_m + gap:
                result[key] = pos_r + (x - pos_m)
        if "a" not in result and a < pos_m + gap + len(ph):
            result["a"] = rs
        if "b" not in result and b <= pos_m + gap + len(ph):
            result["b"] = re_
        pos_m += gap + len(ph)
        pos_r = re_
    for key, x in (("a", a), ("b", b)):
        if key not in result:
            result[key] = pos_r + (x - pos_m)
    return result["a"], result["b"]


rows = []
for r in spans.itertuples():
    u = units.loc[r.unit_id]
    a, b = int(r.span_start), int(r.span_end)
    valid = u["text_masked"][a:b] == r.span_text
    ra = rb = None
    if valid:
        ra, rb = masked_span_to_raw(json.loads(u["mask_edits"]), int(u["char_start"]), a, b)
    rows.append((valid, ra, rb))
spans["span_offset_valid"] = [x[0] for x in rows]
spans["raw_start"] = [x[1] for x in rows]
spans["raw_end"] = [x[2] for x in rows]

invalid = spans[~spans["span_offset_valid"]]
print("Step 8e spans whose offsets do not reproduce their text:", len(invalid))
display(invalid[["unit_id", "span_text", "span_start", "span_end"]].assign(unit_text=lambda d: d["unit_id"].map(units["text_masked"])))

Step 8e spans whose offsets do not reproduce their text: 1


,unit_id,span_text,span_start,span_end,unit_text
7643,row-1416-s007,responsibility,49.0,63.0,With the ability of decision making and sense of responsibility


## 5. Validation

In [5]:
checks = []


def record(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})


record("every Step 8e unit_id exists in the new units", int(joined["text_masked"].isna().sum()) == 0)
record("Step 8e sentence text equals the new unit text (whitespace-collapsed)", int((joined["sentence"].map(compact) != joined["text_masked"].map(compact)).sum()) == 0)
record("every span row received a v0.2 candidate", spans["v02_candidate"].notna().all())
record("candidate codes belong to the schema or to a named review state", set(spans["v02_candidate"]) <= set(schema_df["code"]) | {"E4_or_E5", "needs_review", "none"}, str(sorted(set(spans["v02_candidate"]))))
record("guideline document contains every schema code and no obsolete B1", not missing_in_guide and not (("B1" in guide) or ("Binary Listing" in guide)))

# raw-offset round trip: for units without placeholders the raw slice must equal the span text
bad, tested = [], 0
for r in spans[spans["span_offset_valid"]].itertuples():
    u = units.loc[r.unit_id]
    if u["mask_edits"] == "[]":
        tested += 1
        if raw.loc[u["record_id"], "raw_text"][int(r.raw_start) : int(r.raw_end)] != r.span_text:
            bad.append(r.unit_id)
record("raw[raw_start:raw_end] equals span text for spans in units without placeholders", not bad, f"{tested} spans tested, {len(bad)} mismatches")
unit_with_edit = spans[spans["span_offset_valid"] & spans["unit_id"].map(units["mask_edits"]).ne("[]")]
record("spans in units with placeholders have raw offsets", unit_with_edit["raw_start"].notna().all(), f"{len(unit_with_edit)} spans")
record("Step 8e spans whose offsets reproduce their text", True, f"{int(spans['span_offset_valid'].sum())} of {len(spans)}; the rest are listed above and carry no raw offsets")

validation = pd.DataFrame(checks)
display(validation)
assert validation["passed"].all(), "A validation check failed."

,check,passed,detail
0,every Step 8e unit_id exists in the new units,True,
1,Step 8e sentence text equals the new unit text (whitespace-collapsed),True,
2,every span row received a v0.2 candidate,True,
3,candidate codes belong to the schema or to a named review state,True,"['E1', 'E2', 'E4_or_E5', 'I1', 'I2', 'INC', 'P', 'needs_review', 'none']"
4,guideline document contains every schema code and no obsolete B1,True,
5,raw[raw_start:raw_end] equals span text for spans in units without placeholders,True,"1216 spans tested, 0 mismatches"
6,spans in units with placeholders have raw offsets,True,1 spans
7,Step 8e spans whose offsets reproduce their text,True,1217 of 1218; the rest are listed above and carry no raw offsets


## 6. What changes relative to the earlier Step 8e labels

Counts are span rows, the units they fall in, and distinct unit texts. The same text (for example "Female") appears in many ads, so distinct texts are the more honest measure of variety.

In [6]:
spans["text_masked"] = spans["unit_id"].map(units["text_masked"])
spans["record_id"] = spans["unit_id"].map(units["record_id"])

cross = (
    spans.groupby(["gender_expression_type", "provisional_bias_type", "v02_candidate"])
    .agg(span_rows=("unit_id", "size"), units=("unit_id", "nunique"), distinct_texts=("text_masked", "nunique"), ads=("record_id", "nunique"))
    .reset_index()
    .rename(columns={"provisional_bias_type": "step8e_bias_type"})
)
display(cross)

flags = (
    spans[spans["review_flag"] != ""]
    .groupby(["review_flag", "v02_candidate"])
    .agg(span_rows=("unit_id", "size"), distinct_texts=("text_masked", "nunique"))
    .reset_index()
)
display(flags)

family = {"E1": "explicit", "E2": "explicit", "E3": "explicit", "E4_or_E5": "explicit (pending context)", "I1": "implicit", "I2": "implicit", "INC": "not bias", "P": "not bias", "none": "none (rule rejected)", "needs_review": "pending review"}
spans["v02_family"] = spans["v02_candidate"].map(family)
shift = spans.groupby(["provisional_bias_type", "v02_family"]).size().rename("span_rows").reset_index().rename(columns={"provisional_bias_type": "step8e_bias_type"})
display(shift)

,gender_expression_type,step8e_bias_type,v02_candidate,span_rows,units,distinct_texts,ads
0,binary_listing,unclear,INC,99,99,7,98
1,gender_coded,unclear,I1,350,324,235,240
2,gender_coded,unclear,I2,379,324,219,275
3,gendered_title_pronoun,implicit,E2,37,37,23,29
4,gendered_title_pronoun,unclear,E4_or_E5,5,3,3,2
5,preference,explicit,E1,14,14,6,14
6,preference,explicit,none,1,1,1,1
7,restriction,explicit,E1,303,298,12,298
8,restriction,explicit,E2,3,3,3,3
9,unclear,unclear,P,11,11,7,11


,review_flag,v02_candidate,span_rows,distinct_texts
0,P_candidate_role_noun,I1,63,46
1,P_candidate_role_noun,I2,43,37
2,honorific_candidate,P,11,7
3,needs_human_context,E4_or_E5,5,3
4,needs_human_context,needs_review,16,7
5,reject_candidate_no_gender_word,none,1,1
6,span_narrowing_review,E1,1,1


,step8e_bias_type,v02_family,span_rows
0,explicit,explicit,320
1,explicit,none (rule rejected),1
2,implicit,explicit,37
3,unclear,explicit (pending context),5
4,unclear,implicit,729
5,unclear,not bias,110
6,unclear,pending review,16


### Rows whose label moves between families (review list)

These are the rows where the v0.2 schema and the earlier Step 8e label disagree in kind, or where the rule output is likely to be rejected by a reviewer. They are listed in full (distinct texts) so the effect of each decision can be checked by eye.

In [7]:
moves = spans[
    ((spans["gender_expression_type"] == "gendered_title_pronoun") & (spans["v02_candidate"] == "E2"))
    | (spans["v02_candidate"] == "none")
    | (spans["review_flag"].isin(["P_candidate_role_noun", "honorific_candidate", "needs_human_context", "span_narrowing_review"]))
]
review = (
    moves.groupby(["gender_expression_type", "v02_candidate", "review_flag", "span_text"], dropna=False)
    .agg(span_rows=("unit_id", "size"), example_unit=("unit_id", "first"))
    .reset_index()
    .sort_values(["gender_expression_type", "span_rows"], ascending=[True, False])
)
print(f"{len(review)} distinct (type, candidate, flag, span) combinations; showing the first 60.")
display(review.head(60))

24 distinct (type, candidate, flag, span) combinations; showing the first 60.


,gender_expression_type,v02_candidate,review_flag,span_text,span_rows,example_unit
5,gender_coded,I2,P_candidate_role_noun,Support,34,row-0053-s001
1,gender_coded,I1,P_candidate_role_noun,Analyst,23,row-0064-s001
3,gender_coded,I1,P_candidate_role_noun,analysis,16,row-0093-s004
2,gender_coded,I1,P_candidate_role_noun,Leader,14,row-0335-s001
0,gender_coded,I1,P_candidate_role_noun,Analysis,9,row-0559-s003
6,gender_coded,I2,P_candidate_role_noun,support,9,row-0342-s007
4,gender_coded,I1,P_candidate_role_noun,leader,1,row-1302-s002
7,gendered_title_pronoun,E2,,Foreman,22,row-0131-s001
9,gendered_title_pronoun,E2,,Salesman,10,row-0626-s001
11,gendered_title_pronoun,E2,,salesman,3,row-0948-s003


## 7. Export

In [8]:
cols = [
    "unit_id", "record_id", "component", "text_masked", "span_text", "span_start", "span_end", "span_offset_valid", "raw_start", "raw_end",
    "gender_expression_type", "provisional_bias_type", "gender_direction", "gender_coding", "rule_id",
    "v02_candidate", "v02_family", "review_flag", "crosswalk_reason",
]
spans[cols].to_csv(OUT / "crosswalk_step8e_to_v0.2.csv", index=False, encoding="utf-8-sig")
cross.to_csv(OUT / "crosswalk_summary.csv", index=False, encoding="utf-8-sig")
flags.to_csv(OUT / "crosswalk_flag_summary.csv", index=False, encoding="utf-8-sig")
validation.to_csv(OUT / "validation_checks.csv", index=False, encoding="utf-8-sig")
print("Saved:", *sorted(str(p) for p in OUT.iterdir()), sep="\n  ")

Saved:
  outputs\s4_schema_v0.2\crosswalk_flag_summary.csv
  outputs\s4_schema_v0.2\crosswalk_step8e_to_v0.2.csv
  outputs\s4_schema_v0.2\crosswalk_summary.csv
  outputs\s4_schema_v0.2\sample_ad_view.csv
  outputs\s4_schema_v0.2\sample_candidates.csv
  outputs\s4_schema_v0.2\sample_config.json
  outputs\s4_schema_v0.2\sample_probe_hits.csv
  outputs\s4_schema_v0.2\sample_review_sheet.csv
  outputs\s4_schema_v0.2\sample_validation_checks.csv
  outputs\s4_schema_v0.2\schema_v0.2.json
  outputs\s4_schema_v0.2\validation_checks.csv


## 8. Validation and testing: lexicon-based pre-annotation on a random sample of job ads

**Purpose.** Test whether the v0.2 schema can be applied to real units of our dataset: that lexicon-based pre-annotation produces candidate spans the schema can describe, that every unit of a sampled ad receives a defined status, and which cases the schema or the rules cannot settle. The output is also the review sheet for a human annotator.

**Design**
1. Draw a simple random sample of **60 advertisements** (fixed seed, no stratification) from the preprocessed units. All units of each sampled ad are kept, including units with no gender expression.
2. **Pre-annotation** is the existing lexicon/rule output (Step 8e spans) translated to v0.2 candidates in section 3. No new lexicon is introduced.
3. A **recall probe** scans the sampled units with a deliberately broad list of gender-related terms (gender words, pronouns, honorifics, family and marital terms, *-man / -woman / -lady* endings). A probe hit that no candidate span covers is listed for human review. The probe is a prompt for reviewers, not a label source, and it is noisy by design.
4. Automatic **schema tests** check that candidate codes belong to the schema, that spans lie inside their units, that categories do not overlap within a unit (the guide marks one label per span, longest phrase), and that every unit has exactly one status.

**What this does not show.** No human has labelled the sample, so nothing here measures accuracy, precision, recall or agreement. A sample of 60 ads will contain only a few units per label; rare labels (E3-E5, O) may not appear at all.

In [9]:
SAMPLE_N_ADS = 60
SAMPLE_SEED = 20260101

ads_all = sorted(units["record_id"].unique())
sample_ads = sorted(pd.Series(ads_all).sample(n=SAMPLE_N_ADS, random_state=SAMPLE_SEED))
s_units = units[units["record_id"].isin(sample_ads)].reset_index(drop=True).sort_values("unit_id").reset_index(drop=True)

print(f"{len(ads_all):,} ads in the preprocessed data; sampled {len(sample_ads)} (seed {SAMPLE_SEED}).")
print(f"{len(s_units):,} units in the sample.")
display(s_units["component"].value_counts().rename_axis("component").to_frame("units"))

1,500 ads in the preprocessed data; sampled 60 (seed 20260101).
316 units in the sample.


,units
component,
qualification,256
job_title,60


In [10]:
s_spans = spans[spans["unit_id"].isin(s_units["unit_id"])].copy()

PROBE = re.compile(
    r"\b(?:male|female|males|females|men|women|man|woman|boy|girl|lady|ladies|gentleman|gentlemen|he|she|his|her|him|"
    r"mr|mrs|ms|miss|sir|madam|ma'am|husband|wife|mother|father|pregnant|pregnancy|single|married|lalaki|babae|"
    r"\w+man|\w+men|\w+woman|\w+lady)\b",
    re.I,
)

probe_rows = []
for _, u in s_units.iterrows():
    covered_text = " ".join(s_spans.loc[s_spans["unit_id"] == u["unit_id"], "span_text"].astype(str).str.lower())
    for m in PROBE.finditer(str(u["text_masked"])):
        tok = m.group(0).lower()
        if tok not in covered_text:
            probe_rows.append({"unit_id": u["unit_id"], "component": u["component"], "probe_term": m.group(0), "text_masked": u["text_masked"]})
probe = pd.DataFrame(probe_rows, columns=["unit_id", "component", "probe_term", "text_masked"])

print(f"Candidate spans from the lexicon rules in the sample: {len(s_spans)} rows in {s_spans['unit_id'].nunique()} units, {s_spans['record_id'].nunique()} ads.")
print(f"Recall-probe hits not covered by any candidate span: {len(probe)} in {probe['unit_id'].nunique()} units.")
display(s_spans["v02_candidate"].value_counts().rename_axis("v0.2 candidate").to_frame("span rows"))
display(s_spans[["unit_id", "component", "span_text", "v02_candidate", "review_flag", "text_masked"]].reset_index(drop=True))

Candidate spans from the lexicon rules in the sample: 48 rows in 40 units, 30 ads.
Recall-probe hits not covered by any candidate span: 9 in 7 units.


,span rows
v0.2 candidate,
I2,21
I1,12
E1,10
INC,3
E2,2


,unit_id,component,span_text,v02_candidate,review_flag,text_masked
0,row-0265-s001,job_title,Foreman,E2,,Foreman (Earthworks/Civil)
1,row-0317-s009,qualification,interpersonal,I2,,Good interpersonal skills
2,row-0396-s008,qualification,analyze,I1,,"Ability to input, retrive and analyze data"
3,row-0402-s002,qualification,Female,E1,,Female
4,row-0406-s004,qualification,Responsible,I2,,Responsible for overall functions of distributionship/units pertaining mainly to close coordinat...
5,row-0472-s002,qualification,Female,E1,,Female
6,row-0568-s007,qualification,Analytical,I1,,"Analytical, trustworthy and very responsible"
7,row-0568-s007,qualification,trustworthy,I2,,"Analytical, trustworthy and very responsible"
8,row-0568-s007,qualification,responsible,I2,,"Analytical, trustworthy and very responsible"
9,row-0614-s006,qualification,Honest,I2,,Honest and driven


### Recall-probe hits not covered by a candidate span

Each row is a unit containing a gender-related term for which the rules produced no candidate. Most are expected to be non-bias (for example *human*, *manager*, *single* in another sense); they are listed so a reviewer can see what the rules did not flag, not because they are errors.

In [11]:
display(probe.reset_index(drop=True))
display(probe["probe_term"].str.lower().value_counts().rename_axis("probe term").to_frame("hits"))

,unit_id,component,probe_term,text_masked
0,row-0317-s007,qualification,MS,"Computer literate and proficient in MS Office (MS Word, Excel, PowerPoint)"
1,row-0317-s007,qualification,MS,"Computer literate and proficient in MS Office (MS Word, Excel, PowerPoint)"
2,row-0396-s007,qualification,MS,Proficiency in MS Office
3,row-0472-s003,qualification,Single,Single
4,row-0648-s003,qualification,single,Preferably single
5,row-1165-s001,job_title,Human,Human Resource Manager
6,row-1165-s002,qualification,Human,"Minimum experience of 2 years, Bachelor Degree of Human Resource Development Management"
7,row-1188-s003,qualification,Single,Single or Married
8,row-1188-s003,qualification,Married,Single or Married


,hits
probe term,
ms,3
single,3
human,2
married,1


### Unit status and schema tests

In [12]:
valid_codes = {c[0] for c in SCHEMA}
review_states = {"needs_review", "E4_or_E5", "none"}

def unit_status(uid):
    codes = set(s_spans.loc[s_spans["unit_id"] == uid, "v02_candidate"])
    if codes:
        return "candidate: " + "+".join(sorted(codes))
    if uid in set(probe["unit_id"]):
        return "no candidate, probe hit"
    return "no gender expression found"

s_units["status"] = s_units["unit_id"].map(unit_status)
status_counts = s_units["status"].value_counts().rename_axis("status").to_frame("units")
display(status_counts)

# overlap check on spans whose Step 8e offsets reproduced their text
ok_spans = s_spans[s_spans["span_offset_valid"] == True]
overlaps = []
for uid, g in ok_spans.groupby("unit_id"):
    g = g.sort_values("span_start")
    prev_end, prev_row = None, None
    for _, r in g.iterrows():
        if prev_end is not None and r["span_start"] < prev_end:
            overlaps.append({"unit_id": uid, "span_a": prev_row["span_text"], "code_a": prev_row["v02_candidate"], "span_b": r["span_text"], "code_b": r["v02_candidate"]})
        prev_end, prev_row = r["span_end"], r
overlaps = pd.DataFrame(overlaps, columns=["unit_id", "span_a", "code_a", "span_b", "code_b"])

in_unit = s_spans.apply(lambda r: str(r["span_text"]).strip().lower() in str(r["text_masked"]).lower(), axis=1)
mixed = s_spans.groupby("unit_id")["v02_candidate"].nunique()
sample_checks = []

def srec(name, ok, detail=""):
    sample_checks.append({"check": name, "passed": bool(ok), "detail": detail})

srec("sample has at least 50 ads", len(sample_ads) >= 50, f"{len(sample_ads)} ads")
srec("sample ads are distinct and drawn from the preprocessed units", len(set(sample_ads)) == len(sample_ads) and set(sample_ads) <= set(ads_all))
srec("every unit of each sampled ad is kept", int(units["record_id"].isin(sample_ads).sum()) == len(s_units), f"{len(s_units)} units")
srec("every unit has exactly one status", s_units["status"].notna().all() and s_units["unit_id"].is_unique)
srec("candidate codes belong to the schema or a named review state", set(s_spans["v02_candidate"]) <= valid_codes | review_states, str(sorted(set(s_spans["v02_candidate"]))))
srec("every candidate span text occurs in its unit", bool(in_unit.all()), f"{int((~in_unit).sum())} not found")
srec("no overlapping candidate spans in a unit", len(overlaps) == 0, f"{len(overlaps)} overlaps")
srec("every candidate row has a family", s_spans["v02_family"].notna().all())
sample_validation = pd.DataFrame(sample_checks)
display(sample_validation)
print(f"Units with more than one distinct candidate code: {int((mixed > 1).sum())}")
display(overlaps)

,units
status,
no gender expression found,269
candidate: I2,14
candidate: E1,10
candidate: I1,8
"no candidate, probe hit",7
candidate: I1+I2,3
candidate: INC,3
candidate: E2,2


,check,passed,detail
0,sample has at least 50 ads,True,60 ads
1,sample ads are distinct and drawn from the preprocessed units,True,
2,every unit of each sampled ad is kept,True,316 units
3,every unit has exactly one status,True,
4,candidate codes belong to the schema or a named review state,True,"['E1', 'E2', 'I1', 'I2', 'INC']"
5,every candidate span text occurs in its unit,True,0 not found
6,no overlapping candidate spans in a unit,True,0 overlaps
7,every candidate row has a family,True,


Units with more than one distinct candidate code: 3


,unit_id,span_a,code_a,span_b,code_b


### Ad-level view

An ad is shown as *explicit* or *implicit* candidate only according to the rule-derived families (`counts_as_bias` in the schema). Candidates flagged for review (role-noun, honorific, pronoun context) are counted separately because a reviewer may remove them.

In [13]:
fam = dict(zip(*[[c[0] for c in SCHEMA], [c[2] for c in SCHEMA]]))
bias_codes = {c[0] for c in SCHEMA if c[3]}
s_spans["is_bias_candidate"] = s_spans["v02_candidate"].isin(bias_codes)
s_spans["needs_check"] = s_spans["review_flag"].fillna("").ne("") | s_spans["v02_candidate"].isin(["needs_review", "E4_or_E5"])

ad_rows = []
for ad in sample_ads:
    g = s_spans[(s_spans["record_id"] == ad) & s_spans["is_bias_candidate"]]
    ad_rows.append({
        "record_id": ad,
        "explicit_candidate": bool((g["v02_family"] == "explicit").any()),
        "implicit_candidate": bool((g["v02_family"] == "implicit").any()),
        "only_flagged_candidates": bool(len(g) and g["needs_check"].all()),
        "units_with_candidates": int(g["unit_id"].nunique()),
    })
ad_view = pd.DataFrame(ad_rows)
ad_view["any_bias_candidate"] = ad_view["explicit_candidate"] | ad_view["implicit_candidate"]
summary = pd.DataFrame({
    "ads": [len(ad_view)],
    "any_bias_candidate": [int(ad_view["any_bias_candidate"].sum())],
    "explicit_candidate": [int(ad_view["explicit_candidate"].sum())],
    "implicit_candidate": [int(ad_view["implicit_candidate"].sum())],
    "only_flagged_candidates": [int(ad_view["only_flagged_candidates"].sum())],
})
display(summary)

,ads,any_bias_candidate,explicit_candidate,implicit_candidate,only_flagged_candidates
0,60,28,12,19,0


### Sampled ads with candidates, shown unit by unit

Format: `[component] text   <- code (flag) span`. Ads without any candidate are listed only by id.

In [14]:
by_unit = s_spans.groupby("unit_id")
lines = []
for ad in sample_ads:
    u = s_units[s_units["record_id"] == ad]
    if not u["unit_id"].isin(s_spans["unit_id"]).any():
        continue
    lines.append(f"--- {ad}")
    for _, r in u.iterrows():
        note = ""
        if r["unit_id"] in by_unit.groups:
            g = by_unit.get_group(r["unit_id"])
            note = "   <- " + "; ".join(f"{x.v02_candidate}{' (' + x.review_flag + ')' if x.review_flag else ''} '{x.span_text}'" for x in g.itertuples())
        lines.append(f"[{r['component']}] {r['text_masked']}{note}")
print("\n".join(lines))
print()
print("Ads in the sample with no candidate:", ", ".join(a for a in sample_ads if a not in set(s_spans["record_id"])) or "none")

--- row-0265
[job_title] Foreman (Earthworks/Civil)   <- E2 'Foreman'
--- row-0317
[job_title] Purchasing Staff
[qualification] Graduate of any four year Business Course
[qualification] With knowledge in PEZA Rules is an advantage
[qualification] Can communicate cleraly both in english and filipino
[qualification] Resourceful, flexible, productive and efficient
[qualification] Team player, proactive and cuctomer-oriented
[qualification] Computer literate and proficient in MS Office (MS Word, Excel, PowerPoint)
[qualification] Keen to details
[qualification] Good interpersonal skills   <- I2 'interpersonal'
--- row-0396
[job_title] Creative Services Associate
[qualification] College Graduate
[qualification] 1 to 2 years work experience
[qualification] Knowledgeable in road maps in Luzon (knowledge in VisMin is also desirable)
[qualification] Preferably with experience in dealing with sales agents
[qualification] Preferably with knowledge of warehouse siftware packages like QuickBooks
[q

### Export of the sample and a blank review sheet

`sample_review_sheet.csv` contains every unit of the sampled ads with the rule candidates and empty columns for a human annotator (`human_label`, `human_span`, `notes`). Candidates are shown as a pre-annotation to be curated, as in the reference guide; an annotator may confirm, change or reject them. The sheet contains masked text only.

In [15]:
cand_by_unit = s_spans.groupby("unit_id").apply(
    lambda g: " | ".join(f"{x.v02_candidate}:{x.span_text}" for x in g.itertuples()), include_groups=False
)
sheet = s_units[["unit_id", "record_id", "component", "text_masked", "status"]].copy()
sheet["rule_candidates"] = sheet["unit_id"].map(cand_by_unit).fillna("")
sheet["probe_terms"] = sheet["unit_id"].map(probe.groupby("unit_id")["probe_term"].apply(lambda s: " | ".join(s))).fillna("")
for c in ("human_label", "human_span", "notes"):
    sheet[c] = ""

sheet.to_csv(OUT / "sample_review_sheet.csv", index=False, encoding="utf-8-sig")
s_spans[["unit_id", "record_id", "component", "span_text", "v02_candidate", "v02_family", "review_flag"]].to_csv(OUT / "sample_candidates.csv", index=False, encoding="utf-8-sig")
probe.to_csv(OUT / "sample_probe_hits.csv", index=False, encoding="utf-8-sig")
sample_validation.to_csv(OUT / "sample_validation_checks.csv", index=False, encoding="utf-8-sig")
ad_view.to_csv(OUT / "sample_ad_view.csv", index=False, encoding="utf-8-sig")
json.dump({"n_ads": SAMPLE_N_ADS, "seed": SAMPLE_SEED, "record_ids": sample_ads}, open(OUT / "sample_config.json", "w", encoding="utf-8"), indent=1)
assert sample_validation["passed"].all(), sample_validation[~sample_validation["passed"]]
print("Saved sample outputs to", OUT)

Saved sample outputs to outputs\s4_schema_v0.2
